In [13]:
pip install rank_bm25

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split

# loading datas files
print("loadng datas files")
df_reviews = pd.read_csv('reviews83325.csv', low_memory=False)
df_places = pd.read_csv('Tripadvisor.csv')

# filtering english
print("filtring english reviews")
df_reviews_en = df_reviews[df_reviews['langue'] == 'en'].copy()
df_reviews_en['review'] = df_reviews_en['review'].astype(str)

# group and keep 10 max to fix diff sizes
# we used lambda and head(10) to fix the diff size between places like the subject said
df_reviews_grouped = df_reviews_en.groupby('idplace')['review'].apply(lambda x: ' '.join(x.head(10))).reset_index()

# join
print("merging tables")
df_merged = pd.merge(df_places, df_reviews_grouped, left_on='id', right_on='idplace', how='inner')

# split 50/50
print("spliting train and test")
df_train, df_test = train_test_split(df_merged, test_size=0.5, random_state=42)

loadng datas files
filtring english reviews
merging tables
spliting train and test


In [15]:
from rank_bm25 import BM25Okapi

# init bm25 index
print("init bm25 index on test set")

def tokenize(text):
    return str(text).lower().split()

corpus_test = df_test['review'].tolist()
tokenized_corpus = [tokenize(doc) for doc in corpus_test]

bm25 = BM25Okapi(tokenized_corpus)
print("bm25 model is ready")

init bm25 index on test set
bm25 model is ready


In [16]:
import numpy as np

# startign lvl 1 eval for bm25
print("startign lvl 1 eval for bm25...")
errors_bm25 = []

for index, q_row in df_train.iterrows():
    query_type = q_row['typeR']
    query_text = tokenize(q_row['review'])

    # skip if not testable
    if query_type not in df_test['typeR'].values:
        continue

    scores = bm25.get_scores(query_text)
    ranked_indices = np.argsort(scores)[::-1]

    # find first match
    for position, test_idx in enumerate(ranked_indices):
        test_type = df_test.iloc[test_idx]['typeR']
        if test_type == query_type:
            errors_bm25.append(position)
            break

print(f"bm25 lvl 1 error (mean): {np.mean(errors_bm25):.2f}")

startign lvl 1 eval for bm25...
bm25 lvl 1 error (mean): 0.61


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# tfidf model
print("makin tfidf model...")
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix_test = vectorizer.fit_transform(df_test['review'])
print("tfidf is don")

# eval lvl 1 for tfidf
print("evaluating tfidf lvl 1...")
errors_tfidf_lvl1 = []

for index, q_row in df_train.iterrows():
    query_type = q_row['typeR']
    query_text = q_row['review']

    if query_type not in df_test['typeR'].values:
        continue

    query_vec = vectorizer.transform([query_text])
    similarities = cosine_similarity(query_vec, tfidf_matrix_test).flatten()
    ranked_indices = np.argsort(similarities)[::-1]

    # find first match based purely on reviews
    for position, test_idx in enumerate(ranked_indices):
        test_type = df_test.iloc[test_idx]['typeR']
        if test_type == query_type:
            errors_tfidf_lvl1.append(position)
            break

print(f"tfidf lvl 1 error (mean): {np.mean(errors_tfidf_lvl1):.2f}")

makin tfidf model...
tfidf is don
evaluating tfidf lvl 1...
tfidf lvl 1 error (mean): 0.80


In [18]:
# prep lvl 2 eval
print("prep lvl 2 eval...")

# helper for parsing sets
def extract_categories(val):
    if pd.isna(val):
        return set()
    # list comprehnsion to clean strings and use set for super fast matching
    return set([x.strip().lower() for x in str(val).split(',') if x.strip()])

# check if places match on lvl 2 rules
def match_level_2(q_row, t_row):
    if q_row['typeR'] != t_row['typeR']:
        return False

    t = q_row['typeR']

    if t == 'H':
        if pd.isna(q_row['priceRange']) or pd.isna(t_row['priceRange']):
            return False
        return q_row['priceRange'] == t_row['priceRange']

    elif t == 'R':
        q_cats = extract_categories(q_row['restaurantType']).union(extract_categories(q_row['restaurantTypeCuisine']))
        t_cats = extract_categories(t_row['restaurantType']).union(extract_categories(t_row['restaurantTypeCuisine']))
        # intersection is much fater than double for loops to find at least one match
        return len(q_cats.intersection(t_cats)) > 0

    elif t in ['A', 'AP']:
        q_cats = extract_categories(q_row['activiteSubCategorie']).union(extract_categories(q_row['activiteSubType']))
        t_cats = extract_categories(t_row['activiteSubCategorie']).union(extract_categories(t_row['activiteSubType']))
        return len(q_cats.intersection(t_cats)) > 0

    return False

# runing lvl 2 eval on tfidf
print("runing lvl 2 eval on tfidf...")
errors_tfidf_lvl2 = []

for index, q_row in df_train.iterrows():
    # check if a match exist in the test set
    # we skip query completly if no match exist in test set to optimize calc time
    possible_matches = df_test.apply(lambda t_row: match_level_2(q_row, t_row), axis=1)
    if not possible_matches.any():
        continue

    query_vec = vectorizer.transform([q_row['review']])
    similarities = cosine_similarity(query_vec, tfidf_matrix_test).flatten()
    ranked_indices = np.argsort(similarities)[::-1]

    for position, test_idx in enumerate(ranked_indices):
        test_row = df_test.iloc[test_idx]
        if match_level_2(q_row, test_row):
            errors_tfidf_lvl2.append(position)
            break

print(f"final lvl 2 error tfidf: {np.mean(errors_tfidf_lvl2):.2f}")

prep lvl 2 eval...
runing lvl 2 eval on tfidf...
final lvl 2 error tfidf: 4.14
